# Unidirectional coupling vs. cascaded collisional model

Loads the `LCH_qc_unidirectional_coupling` run (#484) and overlays the measured Q1/Q2
excited-state populations vs. round with the **cascaded coupling** theory ported from
`notebooks/calculator/collisional simulation of dynamics.ipynb` (cell 4): both the discrete
**collisional circuit model** (`Q1_circ`, `Q2_circ`) and the **exact cascaded master
equation** (`Q1_exact`, `Q2_exact`).

The experiment excites `q1`, then each round runs the SWAP-chain `q1→q2` then `q2→q3`
followed by a RESET of `q2` — excitation flows unidirectionally `q1 → q2 → q3` with `q2`
acting as the freshly-reset mediator (the model's environment). Mapping: **Q1 ↔ excite
qubit (q1)**, **Q2 ↔ chain sink (q3)**, **env ↔ reset qubit (q2)**. Q1 decays; Q2 rises then
decays. Tune `gamma1`, `gamma2`, `dt`, `P0` by eye (`theta = sqrt(gamma*dt)` per round;
physical time `t = n*dt`; `P0` = initial excited population).

In [ ]:
# Cell 1 — setup & manual knobs
import os
import json
import sys

import numpy as np
import matplotlib.pyplot as plt
import qutip as qt


def _find_repo_root(start):
    """Walk up from `start` until a directory containing the `scqat` package is found."""
    d = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(d, "scqat")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return None
        d = parent


# scqat is not pip-installed; add the repo root to sys.path so `scqat` resolves
# (same pattern as analysis/try_ramsey_new.py).
_ROOT = _find_repo_root(os.getcwd())
if _ROOT and _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)

from scqat.parsers import load_xarray_h5

_DATA_ROOT = r"D:\SynologyDrive\LiChiehHsiao\AS\SynologyDrive\data\raw_data\2026-06-29"

# gamma1 / gamma2 / dt / P0 are the MANUAL KNOBS you tune by eye:
#   theta_per_round = sqrt(gamma * dt); physical time of round n is t = n * dt
#   P0              = initial excited population of the source qubit (prep/readout infidelity)
RUN = {
    "path": os.path.join(_DATA_ROOT, "#484_LCH_qc_unidirectional_coupling_173944"),
    "gamma1": 1.0, "gamma2": 1.0, "dt": 0.30, "P0": 0.84,
}

In [ ]:
# Cell 2 — load experiment + map qubits (Q1 = excite, env = reset, Q2 = chain sink)
def _model_param(run_dir, key):
    """Read one field from the run's node.json parameters."""
    with open(os.path.join(run_dir, "node.json"), "r", encoding="utf-8") as f:
        node = json.load(f)
    return node["data"]["parameters"]["model"][key]


def experimental_curves(run_dir):
    """Return (rounds, p_Q1, p_Q2, name_Q1, name_Q2) for the unidirectional-coupling run.

    Q1 = the excited (source) qubit, Q2 = the chain sink (the measured qubit that is neither
    the source nor the reset mediator). P(excited) = (state >= 1).mean("shot") — the per-qubit
    marginal population (mirrors marginal_populations in LCHQMDriver _qc_populations.py).
    """
    excite = _model_param(run_dir, "excite_qubit")
    reset = _model_param(run_dir, "reset_qubit")
    ds = load_xarray_h5(os.path.join(run_dir, "ds_raw.h5"))
    measured = [str(q) for q in ds.qubit.values]
    sink = next(q for q in measured if q not in (excite, reset))
    m = (ds["state"] >= 1).mean("shot")
    rounds = ds["round"].values
    return rounds, m.sel(qubit=excite).values, m.sel(qubit=sink).values, excite, sink

In [ ]:
# Cell 3 — cascaded coupling theory (port of the reference notebook cell 4)
_sp, _sm, _s0 = qt.sigmap(), qt.sigmam(), qt.qeye(2)
_ve, _vg = qt.basis(2, 0), qt.basis(2, 1)  # |e> = basis(2,0): rho[0,0] is excited pop
_eg = qt.ket2dm(qt.tensor(_ve, _vg))        # |e g><e g|  (Q1 excited, Q2 ground)
_gg = qt.ket2dm(qt.tensor(_vg, _vg))        # |g g><g g|


def _rho0_sys(P0):
    """Initial 2-qubit (Q1, Q2) state: Q1 excited with probability P0, Q2 ground."""
    return P0 * _eg + (1.0 - P0) * _gg


def cascaded_circuit_populations(rounds, gamma1, gamma2, dt, P0=1.0):
    """Discrete collisional circuit model (cell 4): excitation flows Q1 -> env -> Q2 with a
    fresh (reset) environment each round. Returns (Q1, Q2) excited populations at `rounds`.
    """
    # qubit ordering: 0 = Q1, 1 = Q2, 2 = env
    H1e = qt.tensor(_sm, _s0, _sp) + qt.tensor(_sp, _s0, _sm)
    H2e = qt.tensor(_s0, _sm, _sp) + qt.tensor(_s0, _sp, _sm)
    U1e = qt.Qobj.expm(-1j * np.sqrt(gamma1 * dt) * H1e)
    U2e = qt.Qobj.expm(-1j * np.sqrt(gamma2 * dt) * H2e)
    rho_env = qt.ket2dm(_vg)

    rho_sys = _rho0_sys(P0)  # reduced (Q1, Q2) state
    q1, q2 = [np.real(rho_sys.ptrace(0)[0, 0])], [np.real(rho_sys.ptrace(1)[0, 0])]
    for _ in range(int(np.max(rounds))):
        rho_full = qt.tensor(rho_sys, rho_env)
        rho_sys = (U2e * U1e * rho_full * U1e.dag() * U2e.dag()).ptrace([0, 1])
        q1.append(np.real(rho_sys.ptrace(0)[0, 0]))
        q2.append(np.real(rho_sys.ptrace(1)[0, 0]))
    idx = np.asarray(rounds, dtype=int)
    return np.array(q1)[idx], np.array(q2)[idx]


def cascaded_exact_populations(x_rounds, gamma1, gamma2, dt, P0=1.0):
    """Exact cascaded master equation (cell 4): two local dissipators + the sqrt(g1*g2)
    cascaded cross term. `x_rounds` is a grid in round units; physical time is t = x*dt.
    Returns (Q1_exact, Q2_exact) at those times.
    """
    sp1, sm1 = qt.tensor(_sp, _s0), qt.tensor(_sm, _s0)
    sp2, sm2 = qt.tensor(_s0, _sp), qt.tensor(_s0, _sm)
    L = (gamma1 / 2) * (2 * qt.spre(sm1) * qt.spost(sp1) - qt.spre(sp1 * sm1) - qt.spost(sp1 * sm1))
    L += (gamma2 / 2) * (2 * qt.spre(sm2) * qt.spost(sp2) - qt.spre(sp2 * sm2) - qt.spost(sp2 * sm2))
    L += np.sqrt(gamma1 * gamma2) * (
        qt.spre(sm1) * qt.spost(sp2) + qt.spre(sm2) * qt.spost(sp1)
        - qt.spre(sp2 * sm1) - qt.spost(sp1 * sm2)
    )
    tlist = np.asarray(x_rounds, dtype=float) * dt
    states = qt.mesolve(L, _rho0_sys(P0), tlist).states
    q1 = np.array([np.real(s.ptrace(0)[0, 0]) for s in states])
    q2 = np.array([np.real(s.ptrace(1)[0, 0]) for s in states])
    return q1, q2

In [ ]:
# Cell 4 — one figure: experiment + circuit model + exact master equation
rounds, p_Q1, p_Q2, nQ1, nQ2 = experimental_curves(RUN["path"])
gamma1, gamma2, dt, P0 = RUN["gamma1"], RUN["gamma2"], RUN["dt"], RUN["P0"]

Q1_circ, Q2_circ = cascaded_circuit_populations(rounds, gamma1, gamma2, dt, P0)
x_fine = np.linspace(0, float(np.max(rounds)), 200)
Q1_exact, Q2_exact = cascaded_exact_populations(x_fine, gamma1, gamma2, dt, P0)

run_id = os.path.basename(RUN["path"])
c1, c2 = "C0", "C1"  # colour by qubit; style by source (markers=exp, solid=circ, dashed=exact)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(rounds, p_Q1, "o", color=c1, label=f"Q1 exp ({nQ1})")
ax.plot(rounds, p_Q2, "s", color=c2, label=f"Q2 exp ({nQ2})")
ax.plot(rounds, Q1_circ, "-", color=c1, label="Q1 circuit")
ax.plot(rounds, Q2_circ, "-", color=c2, label="Q2 circuit")
ax.plot(x_fine, Q1_exact, "--", color=c1, alpha=0.7, label="Q1 exact ME")
ax.plot(x_fine, Q2_exact, "--", color=c2, alpha=0.7, label="Q2 exact ME")

ax.set_xlabel("Number of swap-chain + reset rounds")
ax.set_ylabel("P(excited)")
ax.set_title(
    f"{run_id}\n"
    f"Q1={nQ1}, Q2={nQ2}   γ1={gamma1}, γ2={gamma2}, dt={dt}, P0={P0}"
)
ax.legend(ncol=3, fontsize="small")
fig.tight_layout()
plt.show()